#### 1. 1차 필터
- 룰 검출 문장 중 과거 오류 이력 개체명(`EXT_E001`)이 포함된 문장만 추출
- 계엄 문장(`EXT_S001`)은 휴먼 검수 후보에서 제외
- 대상 문장이 하나라도 있는 기사를 `doc_id` 기준으로 원천 데이터에서 추출

In [1]:
import json
from pathlib import Path

import pandas as pd

from src.data_processing.rule_pipeline import load_refer_data
from src.data_processing.prepare.transform import entity_data, sentence_data
from src.data_processing.rules.entity_rules import (
    ext_e001,
    ext_e002,
    ext_e003,
    ext_e004,
    ext_e005,
    ext_e006,
)
from src.data_processing.rules.sentence_rules import ext_s001, ext_s002, ext_s003

# DATA_DIR = Path(r"C:\work\ner\data\org\all_pass_ner_tagged_2")
DATA_DIR = Path(r"C:\work\ner\data\org\all_pass_ner_tagged_2\random_sample_1000")
REPORT_DIR = DATA_DIR / "reports"
input_files = sorted(DATA_DIR.glob("*.json"))
input_files

[WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_2/random_sample_1000/강원일보.json'),
 WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_2/random_sample_1000/경기일보.json'),
 WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_2/random_sample_1000/경향신문.json'),
 WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_2/random_sample_1000/노컷뉴스.json'),
 WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_2/random_sample_1000/뉴스1.json'),
 WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_2/random_sample_1000/뉴스핌.json'),
 WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_2/random_sample_1000/뉴시스.json'),
 WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_2/random_sample_1000/디지털타임스.json'),
 WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_2/random_sample_1000/마이데일리.json'),
 WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_2/random_sample_1000/부산일보.json'),
 WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_2/random_sample_1000/서울신문.json'),
 WindowsPath('C:/work

- 언론사별 1000건 랜덤 샘플링

In [5]:
import random

import ijson

SAMPLE_SIZE = 1000
RANDOM_SEED = 42
SAMPLE_DIR = DATA_DIR / "random_sample_1000"
SAMPLE_DIR.mkdir(exist_ok=True)

def load_source_metadata(input_path):
    source_metadata = {}
    for key in ("id", "metadata"):
        with input_path.open(encoding="utf-8-sig") as file:
            value = next(ijson.items(file, key), None)
        if value is not None:
            source_metadata[key] = value
    return source_metadata

def reservoir_sample_documents(input_path, sample_size, seed):
    rng = random.Random(seed)
    sampled_documents = []

    with input_path.open(encoding="utf-8-sig") as file:
        for index, document in enumerate(ijson.items(file, "document.item")):
            if index < sample_size:
                sampled_documents.append(document)
                continue

            replacement_index = rng.randrange(index + 1)
            if replacement_index < sample_size:
                sampled_documents[replacement_index] = document

    if len(sampled_documents) < sample_size:
        raise ValueError(
            f"{input_path.name}: 문서가 {sample_size}건보다 적습니다 ("
            f"{len(sampled_documents)}건)."
        )
    return sampled_documents


sampled_input_files = []
sampled_id_trace_rows = []
for input_path in input_files:
    press = input_path.stem
    sampled_documents = reservoir_sample_documents(
        input_path,
        SAMPLE_SIZE,
        f"{RANDOM_SEED}:{press}",
    )
    sampled_ids = [document["id"] for document in sampled_documents]
    if len(sampled_ids) != len(set(sampled_ids)):
        raise ValueError(f"{input_path.name}: id가 중복되어 추적할 수 없습니다.")

    sampled_source = load_source_metadata(input_path)
    sampled_source["document"] = sampled_documents
    sampled_path = SAMPLE_DIR / input_path.name
    sampled_path.write_text(
        json.dumps(sampled_source, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    sampled_input_files.append(sampled_path)
    sampled_id_trace_rows.extend(
        {
            "press": press,
            "source_file": str(input_path),
            "sampled_file": str(sampled_path),
            "sampled_rank": rank,
            "id": document_id,
        }
        for rank, document_id in enumerate(sampled_ids, start=1)
    )

sampled_id_trace = pd.DataFrame(sampled_id_trace_rows)
sampled_id_trace.to_csv(
    SAMPLE_DIR / "sampled_id_trace.csv",
    index=False,
    encoding="utf-8-sig",
)
sampled_id_trace

,press,source_file,sampled_file,sampled_rank,id
0,강원일보,C:\work\ner\data\org\all_pass_ner_tagged_2\강원일...,C:\work\ner\data\org\all_pass_ner_tagged_2\ran...,1,NLRW2600000001.2006
1,강원일보,C:\work\ner\data\org\all_pass_ner_tagged_2\강원일...,C:\work\ner\data\org\all_pass_ner_tagged_2\ran...,2,NLRW2600000001.2
2,강원일보,C:\work\ner\data\org\all_pass_ner_tagged_2\강원일...,C:\work\ner\data\org\all_pass_ner_tagged_2\ran...,3,NLRW2600000001.14477
3,강원일보,C:\work\ner\data\org\all_pass_ner_tagged_2\강원일...,C:\work\ner\data\org\all_pass_ner_tagged_2\ran...,4,NLRW2600000001.21036
4,강원일보,C:\work\ner\data\org\all_pass_ner_tagged_2\강원일...,C:\work\ner\data\org\all_pass_ner_tagged_2\ran...,5,NLRW2600000001.8206
...,...,...,...,...,...
18995,충청일보,C:\work\ner\data\org\all_pass_ner_tagged_2\충청일...,C:\work\ner\data\org\all_pass_ner_tagged_2\ran...,996,NLRW2600000004.26978
18996,충청일보,C:\work\ner\data\org\all_pass_ner_tagged_2\충청일...,C:\work\ner\data\org\all_pass_ner_tagged_2\ran...,997,NLRW2600000004.39549
18997,충청일보,C:\work\ner\data\org\all_pass_ner_tagged_2\충청일...,C:\work\ner\data\org\all_pass_ner_tagged_2\ran...,998,NLRW2600000004.16996
18998,충청일보,C:\work\ner\data\org\all_pass_ner_tagged_2\충청일...,C:\work\ner\data\org\all_pass_ner_tagged_2\ran...,999,NLRW2600000004.36774


In [ ]:
# input_files = input_files[1:]
# input_files

[WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_1/경기일보.json'),
 WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_1/경향신문.json')]

In [2]:
def load_reviewed_ids(review_path):
    reviewed_doc_ids = set()
    reviewed_sentence_ids = set()
    with review_path.open(encoding="utf-8") as file:
        for line in file:
            row = json.loads(line)
            if row.get("id"):
                reviewed_doc_ids.add(row["id"])
            sentence_ids = row.get("reverted_sentence_ids", [])
            if not sentence_ids:
                sentence_ids = [
                    sentence.get("sentence_id")
                    for sentence in row.get("original_sentences", [])
                ]
            reviewed_sentence_ids.update(
                sentence_id for sentence_id in sentence_ids if sentence_id
            )
    return reviewed_doc_ids, reviewed_sentence_ids


def get_rule_columns(output_data):
    return [
        column
        for column in output_data.columns
        if column.startswith(("EXT_S", "EXT_E"))
        and not column.endswith("_PATTERN")
    ]


def select_target_documents(source_path, target_doc_ids):
    source = json.loads(source_path.read_text(encoding="utf-8-sig"))
    return {
        **{key: value for key, value in source.items() if key != "document"},
        "document": [
            document for document in source["document"]
            if document["id"] in target_doc_ids
        ],
    }


- 룰 검출 결과 생성
- `run_pipeline()`은 사용하지 않고, 필요한 변환·문장 룰·개체명 룰 함수만 직접 실행합니다.
- 따라서 LLM 입력용 재구성 및 중간 pickle 저장은 수행하지 않습니다.

In [3]:
refer_data = load_refer_data()
error_data = refer_data["error_data"]

error_data = refer_data["error_data"]

for quartile in ("Q4", "Q3", "Q2", "Q1"):
    high_frequency_error_data = error_data.loc[
        error_data["freq_quartile"].eq(quartile)
    ].copy()

    if not high_frequency_error_data.empty:
        selected_quartile = quartile
        break
else:
    raise ValueError(
        "고빈도 과거 오류 개체명이 없습니다. "
        "error_data의 freq_quartile을 확인하세요."
    )

print(f"과거 오류 개체명 기준: {selected_quartile}")

과거 오류 개체명 기준: Q3


In [ ]:
def run_required_rules(input_path):
    output_data = sentence_data(input_path)
    for rule in (ext_s001, ext_s002, ext_s003):
        output_data = rule(output_data)

    output_data = entity_data(output_data)
    output_data = ext_e001(output_data, high_frequency_error_data)
    output_data = ext_e002(output_data)
    output_data = ext_e003(output_data, refer_data["mistag_data"])
    output_data = ext_e004(output_data, refer_data["tag_data"])
    output_data = ext_e005(output_data, refer_data["tag_data"])
    output_data = ext_e006(output_data, refer_data["tag_data"])
    return output_data

# rule_results = {
#     input_path.stem: run_required_rules(input_path)
#     for input_path in input_files
# }

In [ ]:
rule_results = {}

# 1차 실행
for input_path in input_files[:10]:
    print(input_path.stem)
    rule_results[input_path.stem] = run_required_rules(input_path)

# 2차 실행
# for input_path in input_files[10:]:
#     print(input_path.stem)
#     rule_results[input_path.stem] = run_required_rules(input_path)

강원일보
경기일보
경향신문


In [18]:
rule_results # 15분

{'강원일보':          id                doc_id               sentence_id  \
 0      강원일보   NLRW2600000001.2006   NLRW2600000001.2006.1.1   
 1      강원일보   NLRW2600000001.2006   NLRW2600000001.2006.2.1   
 2      강원일보   NLRW2600000001.2006   NLRW2600000001.2006.3.1   
 3      강원일보   NLRW2600000001.2006   NLRW2600000001.2006.4.1   
 4      강원일보   NLRW2600000001.2006   NLRW2600000001.2006.5.1   
 ...     ...                   ...                       ...   
 25494  강원일보  NLRW2600000001.13920  NLRW2600000001.13920.5.4   
 25495  강원일보  NLRW2600000001.13920  NLRW2600000001.13920.5.4   
 25496  강원일보  NLRW2600000001.13920  NLRW2600000001.13920.6.1   
 25497  강원일보  NLRW2600000001.13920  NLRW2600000001.13920.6.1   
 25498  강원일보  NLRW2600000001.13920  NLRW2600000001.13920.6.1   
 
                                                     form  EXT_S001  EXT_S002  \
 0                             길어지는 경기 불황…PC방은 생존 위해 변신 중     False      True   
 1                             벚꽃나무·카페풍 인테리어 ‘데이트 코스’로 인기   

In [25]:
rule_results['강원일보'].doc_id.unique()

<ArrowStringArray>
[ 'NLRW2600000001.2006',     'NLRW2600000001.2', 'NLRW2600000001.14477',
 'NLRW2600000001.21036',  'NLRW2600000001.8206', 'NLRW2600000001.11870',
 'NLRW2600000001.18274',  'NLRW2600000001.2684',  'NLRW2600000001.5197',
 'NLRW2600000001.19663',
 ...
 'NLRW2600000001.24912',  'NLRW2600000001.1818', 'NLRW2600000001.22196',
  'NLRW2600000001.9029', 'NLRW2600000001.12654', 'NLRW2600000001.16221',
 'NLRW2600000001.11712',  'NLRW2600000001.1828', 'NLRW2600000001.18932',
 'NLRW2600000001.13920']
Length: 1000, dtype: str

In [19]:
{press: result.shape for press, result in rule_results.items()}

{'강원일보': (25499, 106),
 '경기일보': (28572, 106),
 '경향신문': (39388, 106),
 '노컷뉴스': (31048, 106),
 '뉴스1': (32667, 106),
 '뉴스핌': (31132, 106),
 '뉴시스': (29305, 106),
 '디지털타임스': (33164, 106),
 '마이데일리': (39814, 106),
 '부산일보': (30780, 106),
 '서울신문': (32956, 106),
 '세계일보': (36942, 106),
 '스포츠경향': (39644, 106),
 '스포츠투데이': (37455, 106),
 '아시아투데이': (31907, 106),
 '이데일리': (36640, 106),
 '이투데이': (34043, 106),
 '일간투데이': (25415, 106),
 '충청일보': (22887, 106)}

In [28]:
def get_target_doc_ids(output_data, review_path):
    rule_columns = get_rule_columns(output_data)
    rule_hit_mask = output_data[rule_columns].eq(True).any(axis=1)
    martial_law_mask = output_data["EXT_S001"].eq(True)
    error_history_mask = output_data["EXT_E001"].eq(True)

    # 룰 검출 건을 문장 단위로 좁힌 뒤, 계엄 문장은 휴먼 검수 후보에서 제외합니다.
    target_sentence_mask = (
        rule_hit_mask & error_history_mask & ~martial_law_mask
    )
    target_sentence_ids = set(
        output_data.loc[target_sentence_mask, "sentence_id"]
    )
    source_doc_count = output_data["doc_id"].nunique()
    source_sentence_count = output_data["sentence_id"].nunique()
    candidate_doc_ids = set(
        output_data.loc[target_sentence_mask, "doc_id"]
    )
    rule_doc_ids = set(output_data.loc[rule_hit_mask, "doc_id"])
    martial_law_doc_ids = set(
        output_data.loc[martial_law_mask, "doc_id"]
    )
    martial_law_sentence_ids = set(
        output_data.loc[martial_law_mask, "sentence_id"]
    )
    error_history_doc_ids = set(
        output_data.loc[error_history_mask, "doc_id"]
    )

    reviewed_doc_ids, reviewed_sentence_ids = load_reviewed_ids(review_path)
    reviewed_candidate_doc_ids = candidate_doc_ids & reviewed_doc_ids

    return {
        "rule_doc_ids": rule_doc_ids,
        "error_history_doc_ids": error_history_doc_ids,
        "candidate_doc_ids": candidate_doc_ids,
        "target_sentence_ids": target_sentence_ids,
        "target_sentence_count": len(target_sentence_ids),
        "source_doc_count": source_doc_count,
        "source_sentence_count": source_sentence_count,
        "martial_law_doc_ids": martial_law_doc_ids,
        "martial_law_sentence_ids": martial_law_sentence_ids,
        "reviewed_doc_ids": reviewed_doc_ids,
        "reviewed_sentence_ids": reviewed_sentence_ids,
        "reviewed_candidate_doc_ids": reviewed_candidate_doc_ids,
    }

doc_ids_by_press = {}
for press, output_data in rule_results.items():
    review_path = REPORT_DIR / press / "review_needed.jsonl"
    doc_ids_by_press[press] = get_target_doc_ids(output_data, review_path)

for ids in doc_ids_by_press.values():
    assert ids["candidate_doc_ids"] <= ids["rule_doc_ids"]
    assert ids["candidate_doc_ids"] <= ids["error_history_doc_ids"]
    assert not (ids["target_sentence_ids"] & ids["martial_law_sentence_ids"])

review_summary = pd.DataFrame([
    {
        "press": press,
        "source_articles": ids["source_doc_count"],
        "source_sentences": ids["source_sentence_count"],
        "rule_articles_detected": len(ids["rule_doc_ids"]),
        "martial_law_articles_excluded": len(ids["martial_law_doc_ids"]),
        "martial_law_article_ratio": (
            len(ids["martial_law_doc_ids"]) / ids["source_doc_count"]
            if ids["source_doc_count"]
            else 0
        ),
        "martial_law_article_pct": (
            len(ids["martial_law_doc_ids"]) * 100 / ids["source_doc_count"]
            if ids["source_doc_count"]
            else 0
        ),
        "target_sentences": ids["target_sentence_count"],
        "error_history_articles": len(ids["error_history_doc_ids"]),
        "candidate_articles": len(ids["candidate_doc_ids"]),
        "candidate_article_ratio": (
            len(ids["candidate_doc_ids"]) / ids["source_doc_count"]
            if ids["source_doc_count"]
            else 0
        ),
        "candidate_article_pct": (
            len(ids["candidate_doc_ids"]) * 100 / ids["source_doc_count"]
            if ids["source_doc_count"]
            else 0
        ),
        "target_sentence_ratio": (
            ids["target_sentence_count"] / ids["source_sentence_count"]
            if ids["source_sentence_count"]
            else 0
        ),
        "target_sentence_pct": (
            ids["target_sentence_count"] * 100 / ids["source_sentence_count"]
            if ids["source_sentence_count"]
            else 0
        ),
        "source_reviewed_candidate_articles": len(ids["reviewed_candidate_doc_ids"]),
        "reviewed_ratio": (
            len(ids["reviewed_candidate_doc_ids"])
            / len(ids["candidate_doc_ids"])
            if ids["candidate_doc_ids"]
            else 0
        ),
        "reviewed_pct": (
            len(ids["reviewed_candidate_doc_ids"]) * 100
            / len(ids["candidate_doc_ids"])
            if ids["candidate_doc_ids"]
            else 0
        ),
    }
    for press, ids in doc_ids_by_press.items()
])
overall = review_summary.sum(numeric_only=True)
overall["press"] = "전체"
overall["candidate_article_ratio"] = (
    overall["candidate_articles"] / overall["source_articles"]
    if overall["source_articles"]
    else 0
)
overall["candidate_article_pct"] = overall["candidate_article_ratio"] * 100
overall["martial_law_article_ratio"] = (
    overall["martial_law_articles_excluded"] / overall["source_articles"]
    if overall["source_articles"]
    else 0
)
overall["martial_law_article_pct"] = overall["martial_law_article_ratio"] * 100
overall["target_sentence_ratio"] = (
    overall["target_sentences"] / overall["source_sentences"]
    if overall["source_sentences"]
    else 0
)
overall["target_sentence_pct"] = overall["target_sentence_ratio"] * 100
overall["source_reviewed_candidate_articles"] = sum(
    len(ids["reviewed_candidate_doc_ids"])
    for ids in doc_ids_by_press.values()
)
overall["reviewed_ratio"] = (
    overall["source_reviewed_candidate_articles"] / overall["candidate_articles"]
    if overall["candidate_articles"]
    else 0
)
overall["reviewed_pct"] = overall["reviewed_ratio"] * 100
review_summary = pd.concat([review_summary, pd.DataFrame([overall])], ignore_index=True)

martial_law_article_ids = pd.DataFrame([
    {"press": press, "doc_id": doc_id}
    for press, ids in doc_ids_by_press.items()
    for doc_id in sorted(ids["martial_law_doc_ids"])
])
reviewed_id_trace = pd.DataFrame([
    {
        "press": press,
        "doc_id": doc_id,
        "reviewed": doc_id in ids["reviewed_candidate_doc_ids"],
    }
    for press, ids in doc_ids_by_press.items()
    for doc_id in sorted(ids["candidate_doc_ids"])
])

display(review_summary)
display(martial_law_article_ids)

,press,source_articles,source_sentences,rule_articles_detected,martial_law_articles_excluded,martial_law_article_ratio,martial_law_article_pct,target_sentences,error_history_articles,candidate_articles,candidate_article_ratio,candidate_article_pct,target_sentence_ratio,target_sentence_pct,source_reviewed_candidate_articles,reviewed_ratio,reviewed_pct
0,강원일보,1000.0,11458.0,972.0,38.0,0.038000,3.800000,476.0,248.0,247.0,0.247000,24.700000,0.041543,4.154303,19,0.076923,7.692308
1,경기일보,1000.0,12238.0,961.0,46.0,0.046000,4.600000,473.0,249.0,248.0,0.248000,24.800000,0.038650,3.865011,24,0.096774,9.677419
2,경향신문,1000.0,16712.0,983.0,77.0,0.077000,7.700000,1279.0,386.0,386.0,0.386000,38.600000,0.076532,7.653183,59,0.152850,15.284974
3,노컷뉴스,1000.0,13315.0,970.0,35.0,0.035000,3.500000,832.0,307.0,306.0,0.306000,30.600000,0.062486,6.248592,36,0.117647,11.764706
4,뉴스1,1000.0,13742.0,986.0,31.0,0.031000,3.100000,943.0,340.0,340.0,0.340000,34.000000,0.068622,6.862174,41,0.120588,12.058824
5,뉴스핌,1000.0,13694.0,978.0,25.0,0.025000,2.500000,993.0,372.0,370.0,0.370000,37.000000,0.072514,7.251351,47,0.127027,12.702703
6,뉴시스,1000.0,12995.0,990.0,22.0,0.022000,2.200000,807.0,352.0,352.0,0.352000,35.200000,0.062101,6.210081,35,0.099432,9.943182
7,디지털타임스,1000.0,14436.0,973.0,43.0,0.043000,4.300000,1081.0,370.0,370.0,0.370000,37.000000,0.074882,7.488224,52,0.140541,14.054054
8,마이데일리,1000.0,16373.0,992.0,0.0,0.000000,0.000000,732.0,371.0,371.0,0.371000,37.100000,0.044708,4.470775,57,0.153639,15.363881
9,부산일보,1000.0,13591.0,988.0,27.0,0.027000,2.700000,765.0,315.0,315.0,0.315000,31.500000,0.056287,5.628725,36,0.114286,11.428571


,press,doc_id
0,강원일보,NLRW2600000001.1015
1,강원일보,NLRW2600000001.10511
2,강원일보,NLRW2600000001.12732
3,강원일보,NLRW2600000001.12840
4,강원일보,NLRW2600000001.16204
...,...,...
525,일간투데이,NWRW2600000005.1119
526,일간투데이,NWRW2600000005.18611
527,일간투데이,NWRW2600000005.43810
528,충청일보,NLRW2600000004.1438


- 룰 검출 기사를 제외하지 않고, 룰 검출 문장 중 휴먼 검수 후보 문장을 선별
- `EXT_E001`이면서 계엄(`EXT_S001`)이 아닌 문장을 포함한 기사만 유지
- 원천 `review_needed.jsonl`은 전처리/복원 검수 이력으로 별도 집계하며, 2차 LLM 오류 결과와 혼동하지 않음

In [29]:
filtered_articles = {
    press: select_target_documents(
        DATA_DIR / f"{press}.json",
        ids["candidate_doc_ids"],
    )
    for press, ids in doc_ids_by_press.items()
}

article_counts = {
    press: len(data["document"])
    for press, data in filtered_articles.items()
}
article_counts

{'강원일보': 247,
 '경기일보': 248,
 '경향신문': 386,
 '노컷뉴스': 306,
 '뉴스1': 340,
 '뉴스핌': 370,
 '뉴시스': 352,
 '디지털타임스': 370,
 '마이데일리': 371,
 '부산일보': 315,
 '서울신문': 359,
 '세계일보': 365,
 '스포츠경향': 322,
 '스포츠투데이': 300,
 '아시아투데이': 367,
 '이데일리': 407,
 '이투데이': 465,
 '일간투데이': 205,
 '충청일보': 207}

In [26]:
DATA_DIR

WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_2/random_sample_1000')

In [30]:
# 필요하면 언론사별 필터 결과를 JSON으로 저장합니다.
OUTPUT_DIR = DATA_DIR / "filtered_by_rule_pipeline"
OUTPUT_DIR.mkdir(exist_ok=True)

for press, data in filtered_articles.items():
    output_path = OUTPUT_DIR / f"{press}.json"
    output_path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

review_summary.to_csv(OUTPUT_DIR / "review_summary.csv", index=False, encoding="utf-8-sig")
martial_law_article_ids.to_csv(OUTPUT_DIR / "martial_law_article_ids.csv", index=False, encoding="utf-8-sig")
reviewed_id_trace.to_csv(OUTPUT_DIR / "reviewed_id_trace.csv", index=False, encoding="utf-8-sig")

OUTPUT_DIR

WindowsPath('C:/work/ner/data/org/all_pass_ner_tagged_2/random_sample_1000/filtered_by_rule_pipeline')

In [33]:
# 언론사별 필터링 결과를 한 번에 확인합니다.
import re

RESULT_DIR = Path(r"C:\work\ner\data\org\all_pass_ner_tagged_2\random_sample_1000\filtered_by_rule_pipeline")
available_press = sorted(path.stem for path in RESULT_DIR.glob("*.json"))
summary_rows = []

for press in available_press:
    data = json.loads((RESULT_DIR / f"{press}.json").read_text(encoding="utf-8"))
    documents = data.get("document", [])
    paragraphs = [
        paragraph
        for document in documents
        for paragraph in document.get("paragraph", [])
    ]
    sentences = [
        sentence.get("form", "")
        for paragraph in paragraphs
        for sentence in paragraph.get("sentence", [])
    ]
    tag_count = sum(len(re.findall(r"<[^<>]*?:\s*[A-Z]{2}>", form)) for form in sentences)
    tagged_sentence_count = sum(bool(re.search(r"<[^<>]*?:\s*[A-Z]{2}>", form)) for form in sentences)
    summary_rows.append({
        "언론사": press,
        "문서 수": len(documents),
        "문단 수": len(paragraphs),
        "문장 수": len(sentences),
        "태그 포함 문장 수": tagged_sentence_count,
        "태그 수": tag_count,
    })

summary = (
    pd.DataFrame(summary_rows)
    .sort_values("문서 수", ascending=False)
    .reset_index(drop=True)
)

# 계엄 기사는 필터링 결과에서 제외되므로 별도 목록에서 언론사별로 집계합니다.
martial_law_path = RESULT_DIR / "martial_law_article_ids.csv"
if martial_law_path.exists():
    martial_law_counts = (
        pd.read_csv(martial_law_path, encoding="utf-8-sig")
        .groupby("press")
        .size()
        .rename("계엄 기사 수")
    )
    summary = summary.merge(
        martial_law_counts,
        left_on="언론사",
        right_index=True,
        how="left",
    )
else:
    summary["계엄 기사 수"] = 0

summary["계엄 기사 수"] = summary["계엄 기사 수"].fillna(0).astype(int)
summary["계엄 기사 비율(%)"] = (
    summary["계엄 기사 수"] / (summary["문서 수"] + summary["계엄 기사 수"]) * 100
).round(2)
summary


,언론사,문서 수,문단 수,문장 수,태그 포함 문장 수,태그 수,계엄 기사 수,계엄 기사 비율(%)
0,이투데이,465,4609,7498,5743,16257,31,6.25
1,이데일리,407,4434,7021,5312,15310,30,6.86
2,경향신문,386,3868,6857,5413,16153,77,16.63
3,마이데일리,371,3765,6911,5552,16485,0,0.00
4,뉴스핌,370,3921,5620,4382,12594,25,6.33
5,디지털타임스,370,4016,5898,4535,13399,43,10.41
6,아시아투데이,367,3745,5890,4549,12977,13,3.42
7,세계일보,365,3924,6381,4863,14766,58,13.71
8,서울신문,359,3358,5552,4226,12923,48,11.79
9,뉴시스,352,4074,4996,3949,11613,22,5.88


In [ ]:
SAMPLE_SIZE = 1000

summary["문서 비율(1000건 기준)(%)"] = (
    summary["문서 수"] / SAMPLE_SIZE * 100
).round(2)

In [36]:
summary_total = {
    "언론사": "총계",
    "문서 수": summary["문서 수"].sum(),
    "문단 수": summary["문단 수"].sum(),
    "문장 수": summary["문장 수"].sum(),
    "태그 포함 문장 수": summary["태그 포함 문장 수"].sum(),
    "태그 수": summary["태그 수"].sum(),
    "계엄 기사 수": summary["계엄 기사 수"].sum(),
}

summary_total["계엄 기사 비율(%)"] = round(
    summary_total["계엄 기사 수"]
    / (summary_total["문서 수"] + summary_total["계엄 기사 수"])
    * 100,
    2,
)

summary_with_total = pd.concat(
    [summary, pd.DataFrame([summary_total])],
    ignore_index=True,
)

summary_with_total

,언론사,문서 수,문단 수,문장 수,태그 포함 문장 수,태그 수,계엄 기사 수,계엄 기사 비율(%),문서 비율(1000건 기준)(%)
0,이투데이,465,4609,7498,5743,16257,31,6.25,46.5
1,이데일리,407,4434,7021,5312,15310,30,6.86,40.7
2,경향신문,386,3868,6857,5413,16153,77,16.63,38.6
3,마이데일리,371,3765,6911,5552,16485,0,0.00,37.1
4,뉴스핌,370,3921,5620,4382,12594,25,6.33,37.0
5,디지털타임스,370,4016,5898,4535,13399,43,10.41,37.0
6,아시아투데이,367,3745,5890,4549,12977,13,3.42,36.7
7,세계일보,365,3924,6381,4863,14766,58,13.71,36.5
8,서울신문,359,3358,5552,4226,12923,48,11.79,35.9
9,뉴시스,352,4074,4996,3949,11613,22,5.88,35.2
